# Weight Probing

Probe weight matrices for structure without running inference:
spectral signatures, role specialization, weight-activation alignment,
pruning sensitivity, and weight geometry.

## Why This Matters

Probing weight tests what information is linearly accessible in model representations. Linear probes provide a principled way to measure whether specific concepts (syntax, semantics, factual knowledge) are encoded at each layer.

**Key references:**
- [Belinkov (2022) "Probing Classifiers"](https://arxiv.org/abs/2102.12452) — Methodology for probing neural representations

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.weight_probing import (
    spectral_signatures,
    role_specialization,
    weight_activation_alignment,
    pruning_sensitivity,
    weight_geometry,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30])
print("Model ready")

## Spectral Signatures

In [ ]:
result = spectral_signatures(model, layer=0, component='attn')
for name, info in result.items():
    print(f"{name}: rank={info['effective_rank']:.1f}, entropy={info['spectral_entropy']:.3f}, "
          f"cond={info['condition_number']:.1f}")

## Role Specialization

In [ ]:
for l in range(min(2, cfg.n_layers)):
    result = role_specialization(model, layer=l)
    scores = [f'{float(s):.3f}' for s in result['specialization_scores']]
    print(f"Layer {l}: specialization={scores}, most_unique=H{result['most_unique_head']}")

## Weight-Activation Alignment

In [ ]:
for h in range(cfg.n_heads):
    result = weight_activation_alignment(model, tokens, layer=0, head=h)
    print(f"L0H{h}: Q={result.get('q_alignment', 0):.3f}, K={result.get('k_alignment', 0):.3f}, "
          f"V={result.get('v_alignment', 0):.3f}, OV={result.get('ov_alignment', 0):.3f}")

## Pruning Sensitivity

In [ ]:
result = pruning_sensitivity(model, layer=0, component='attn')
print(f"Importance scores: {[f'{float(s):.4f}' for s in result['importance_scores']]}")
print(f"Pruning order: {result['pruning_order']}")
print(f"Cumulative norm loss: {[f'{float(c):.2f}' for c in result['cumulative_norm_loss']]}")

## Weight Geometry

In [ ]:
result = weight_geometry(model)
for prof in result['norm_profile']:
    print(f"Layer {prof['layer']}: total_norm={prof['total']:.3f}")
print(f"\nIsotropy: {[f'{float(i):.3f}' for i in result['isotropy_profile']]}")